## File Import

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output

In [ ]:
file=r"Path/File_Attributes.csv"

In [3]:
df=pd.read_csv(file, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
# df=df.drop_duplicates()
# df = df[df['PartTerminologyName'].notna()]
# df

In [ ]:
df

In [5]:
df['FileName']=df.Brand+"_"+df.PartTerminologyName

In [ ]:
df['Product Group'].unique()

In [7]:
df = df[df['Parent-Child- Customer Brand'] == "Parent"]

In [8]:
output_location=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\FBG_Data\Phase-2\FBG_Circulation"

In [ ]:
df['Product Group'].unique()

In [ ]:
import re
#filtering only Required Product Group
Type="Brakes"
df_IN=df[df['Product Group']==Type]
Filenames=df_IN.FileName.unique()
length=len(Filenames)
for i in tqdm(range (length)):
    BrandName=df[(df.FileName==Filenames[i])]['BrandName'].mode()[0]
    print(i, Filenames[i])
    # and separating Autocare Attributes and FBG Attributes
    df1=df_IN[(df_IN.FileName==Filenames[i]) & (df_IN['Attribute_Type']=="Autocare_Attribute")]
    df1=df1.drop(columns=['FileName'])    
    df1=df1[['PartNumber','PartTerminologyName', 'PAName', 'Value']]
    grouped=df1.groupby(['PartNumber','PartTerminologyName', 'PAName'])['Value'].agg(lambda x: list(x) if len(x) > 1 else x.iloc[0])
    # Unstack to get pivot-style format
    pivot_df_AC = grouped.unstack(fill_value=None)
    df2=df_IN[(df_IN.FileName==Filenames[i]) & (df_IN['Attribute_Type']!="Autocare_Attribute")]
    df2=df2.drop(columns=['FileName'])
    df2=df2[['PartNumber','PartTerminologyName', 'PAName', 'Value']]
    grouped=df2.groupby(['PartNumber','PartTerminologyName', 'PAName'])['Value'].agg(lambda x: list(x) if len(x) > 1 else x.iloc[0])
    # Unstack to get pivot-style format
    pivot_df_FBG= grouped.unstack(fill_value=None)
    Filename=re.sub(r"[^a-zA-Z0-9\s_]", "", Filenames[i].split("_")[1])
    with pd.ExcelWriter(rf"{output_location}\{Type}\\"+BrandName+"_"+Filename.replace(" ","")+".xlsx") as writer:
        pivot_df_AC.to_excel(writer, sheet_name="Autocare", index=True, engine='openpyxl')
        pivot_df_FBG.to_excel(writer, sheet_name="FBG", index=True, engine='openpyxl')
    #print("File FBG_SKU_Attributes_"+Filenames[i]+".xlsx created successfully")
    clear_output(wait=True)

In [ ]:
df1

In [40]:
df_IN=df[df['Product Group']==Type]
len(df_IN.PartTerminologyName.unique())

73

In [ ]:
Brand_Filenames=df.Brand.unique()
print(len(Brand_Filenames))
for i in range (len(Brand_Filenames)):
    print(i)
    df1=df[df.Brand==Brand_Filenames[i]]
    df1=df1.drop(columns=['Brand'])
    ProductNames=df1.PartTerminologyName.unique()
    FileName=rf"{output_location}\FBG_SKU_Attributes_"+Brand_Filenames[i]+".xlsx"
    # df1=df1[['PartNumber', 'PAName', 'Value']]
        # Create Excel writer per region
    with pd.ExcelWriter(FileName, engine='xlsxwriter') as writer:
        for j in range(len(ProductNames)):
            df2=df1[df1.PartTerminologyName==ProductNames[j]]
            # df2=df2[['PartNumber', 'PAName', 'Value']]
            grouped=df2.groupby(['PartTerminologyName','PartNumber', 'PAName'])['Value'].agg(lambda x: list(x) if len(x) > 1 else x.iloc[0])
            # Unstack to get pivot-style format
            pivot_df = grouped.unstack(fill_value=None)
            print("Processing Product:", ProductNames[j])
            ProductNames[j]=ProductNames[j].replace('/', '_').replace(' ', "")
            if len(ProductNames[j]) > 31:
                ProductNames[j] = ProductNames[j][:31]
            pivot_df.to_excel(writer, sheet_name=str(ProductNames[j]), index=True)
    print("File FBG_SKU_Attributes_"+Brand_Filenames[i]+".xlsx created successfully")
    clear_output(wait=True)
        